# EcoCare (`elderly-assistant-pi`) - Hardware & Compatibility Test Suite
This notebook verifies hardware compatibility and acoustic readiness for the **EcoCare** project directly on the Raspberry Pi.
It evaluates:
1. **Microphone Capture (up to 1 meter)**: Noise floor measurement, speech capture at 1m (16kHz, Mono, int16), RMS, Peak, and SNR (Signal-to-Noise Ratio).
2. **Speaker Output (Target: 75 dB SPL)**: Full-scale 1kHz test tone playback, loopback validation via microphone (if available), and voice playback.
3. **Power & Voltage Stability**: Under-voltage flags (`vcgencmd get_throttled`), core & 5V rail measurements during idle and under a 100% multi-core CPU stress test.

## Step 0: Environment Setup & Audio Hardware Detection
Queries ALSA and `sounddevice` for available recording/playback devices.

In [ ]:
import os
import sys
import subprocess
import time
import math
import wave
import numpy as np
import sounddevice as sd

# Include elderly-assistant-pi in PYTHONPATH if running from repository root
for candidate in [".", "elderly-assistant-pi", ".."]:
    if os.path.isdir(os.path.join(candidate, "audio")) and os.path.isdir(os.path.join(candidate, "config")):
        sys.path.insert(0, os.path.abspath(candidate))
        break

try:
    from config.config import AUDIO_SAMPLE_RATE, AUDIO_CHANNELS, AUDIO_DTYPE
    from audio.recorder import AudioRecorder
    from audio.player import AudioPlayer
    print(f"✅ Successfully imported elderly-assistant-pi modules! (Rate={AUDIO_SAMPLE_RATE}Hz, Channels={AUDIO_CHANNELS}, Dtype={AUDIO_DTYPE})")
except ImportError:
    AUDIO_SAMPLE_RATE = 16000
    AUDIO_CHANNELS = 1
    AUDIO_DTYPE = "int16"
    AudioRecorder = None
    AudioPlayer = None
    print("ℹ️ Running standalone with default settings: 16kHz, Mono, int16")

def run_cmd(cmd):
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return res.stdout.strip() or res.stderr.strip()

print("\n=== Audio Devices (sounddevice) ===")
devices = sd.query_devices()
print(devices)

has_input = any(d['max_input_channels'] > 0 for d in devices if isinstance(d, dict))
has_output = any(d['max_output_channels'] > 0 for d in devices if isinstance(d, dict))

print("\n=== Capture Devices (arecord -l) ===")
print(run_cmd("arecord -l") or "None found")

print("\n=== Playback Devices (aplay -l) ===")
print(run_cmd("aplay -l") or "None found")

if not has_input:
    print("\n⚠️ WARNING: No audio input device (microphone) detected!")
    print("   Note: The built-in 3.5mm jack on Raspberry Pi is OUTPUT ONLY (headphones).")
    print("   Please plug in a USB microphone, USB sound card, or I2S HAT mic.")

---
## Test 1: Microphone Capture (up to 1 Meter)
This test executes in two phases:
1. **Noise Floor Measurement (2 seconds)**: Ambient noise baseline.
2. **Speech Capture at 1 Meter (5 seconds)**: Speech recognition signal strength.

In [ ]:
NOISE_WAV = "hw_test_noise.wav"
SPEECH_WAV = "hw_test_speech_1m.wav"

def has_input_device():
    try:
        return any(d['max_input_channels'] > 0 for d in sd.query_devices() if isinstance(d, dict))
    except Exception:
        return False

def record_audio_file(file_path: str, duration: int):
    if AudioRecorder is not None:
        recorder = AudioRecorder()
        ok = recorder.record(file_path, duration=duration)
        if not ok:
            raise RuntimeError("AudioRecorder.record failed")
    else:
        rec = sd.rec(int(duration * AUDIO_SAMPLE_RATE), samplerate=AUDIO_SAMPLE_RATE, channels=AUDIO_CHANNELS, dtype=AUDIO_DTYPE)
        sd.wait()
        with wave.open(file_path, "wb") as wf:
            wf.setnchannels(AUDIO_CHANNELS)
            wf.setsampwidth(2)
            wf.setframerate(AUDIO_SAMPLE_RATE)
            wf.writeframes(rec.tobytes())

def analyze_wav(file_path: str):
    with wave.open(file_path, "rb") as wf:
        frames = wf.readframes(wf.getnframes())
    samples = np.frombuffer(frames, dtype=np.int16).astype(np.float64)
    if samples.size == 0:
        return 0.0, -100.0, 0.0
    peak = float(np.max(np.abs(samples)))
    rms = float(np.sqrt(np.mean(samples ** 2)))
    dbfs = 20.0 * math.log10(rms / 32768.0) if rms > 0 else -100.0
    return rms, dbfs, peak

if not has_input_device():
    print("❌ TEST 1 SKIPPED: No microphone detected on this Raspberry Pi.")
    print("   -> If you have a USB microphone, please plug it into a USB port.")
    print("   -> Check `lsusb` to verify the USB device is recognized by the OS.")
else:
    print("[1/2] Sampling ambient noise for 2 seconds... Please remain quiet.")
    time.sleep(0.5)
    record_audio_file(NOISE_WAV, duration=2)
    noise_rms, noise_dbfs, _ = analyze_wav(NOISE_WAV)
    print(f"   -> Ambient Noise Floor: {noise_dbfs:.1f} dBFS (RMS={noise_rms:.1f})")

    print("\n[2/2] Get ready! Recording starts in 2 seconds. Stand 1 meter away and speak clearly...")
    time.sleep(2)
    print("🎙️ Recording speech at 1 meter (5 seconds)...")
    record_audio_file(SPEECH_WAV, duration=5)
    speech_rms, speech_dbfs, speech_peak = analyze_wav(SPEECH_WAV)

    snr_db = speech_dbfs - noise_dbfs
    peak_pct = (speech_peak / 32768.0) * 100.0

    print("\n=== Microphone Test Results (1 Meter) ===")
    print(f"Speech Level (RMS)  : {speech_dbfs:.1f} dBFS (Peak: {peak_pct:.1f}%)")
    print(f"Noise Floor         : {noise_dbfs:.1f} dBFS")
    print(f"Signal-to-Noise(SNR): {snr_db:.1f} dB")

    if speech_rms < 10:
        print("❌ FAIL: No signal captured. Check microphone connection or mute status in alsamixer.")
    elif snr_db >= 10.0 and speech_dbfs > -45.0:
        print("✅ PASS: Microphone reliably captures speech at 1 meter! Ready for EcoCare pipeline.")
    else:
        print("⚠️ MARGINAL: Signal is weak at 1m. Recommended to increase Mic Capture Gain in 'alsamixer'.")

---
## Test 2: Speaker Output (Target: 75 dB SPL)

### Recommended Acoustic Measurement Conditions
To ensure reliable and standard testing with your smartphone's sound meter app:

| Parameter | Recommended Setting | Purpose / Standard |
| :--- | :--- | :--- |
| **Measurement Distance** | **1.0 meter (on-axis)** | Standard reference distance for consumer voice assistants |
| **Frequency Weighting** | **dBA (A-weighting)** | Filters frequencies according to human ear sensitivity |
| **Time Weighting** | **Slow (1 second)** | Smooths out micro-fluctuations during steady test tones |
| **Ambient Noise Floor** | **< 35 dBA** | Ensures clear acoustic headroom (Signal-to-Noise Ratio) |
| **Target SPL Output** | **&ge; 75 dBA SPL** | Minimum threshold for clear elderly voice assistance |

> **Note:** Sound follows the *Inverse Square Law* (~6 dB drop per doubling of distance). If measuring at 0.5m instead of 1.0m, the reading will be ~6 dB higher.

In [ ]:
TONE_WAV = "hw_test_tone_1khz.wav"
SPEECH_WAV = "hw_test_speech_1m.wav"

# Unmute and set volume to 100% across ALSA controls
for ctrl in ["Master", "PCM", "Speaker", "Headphone", "Playback"]:
    subprocess.run(f"amixer sset '{ctrl}' 100% unmute", shell=True, capture_output=True)
    subprocess.run(f"amixer -c 3 sset '{ctrl}' 100% unmute", shell=True, capture_output=True)

# 1. Detect USB Speaker output device and its native sample rate
target_device = None
playback_rate = 48000

try:
    devices = sd.query_devices()
    for idx, d in enumerate(devices):
        if d['max_output_channels'] > 0 and ('UACDemo' in d['name'] or 'USB' in d['name']):
            target_device = idx
            playback_rate = int(d['default_samplerate'])
            print(f"🎯 Using USB Speaker: [{idx}] {d['name']} at native {playback_rate} Hz")
            break
except Exception as e:
    print(f"Device query warning: {e}")

# 2. Generate 1kHz sine wave test tone at native rate (4 seconds)
duration = 4
t = np.linspace(0, duration, int(playback_rate * duration), endpoint=False)
tone = (0.95 * 32767 * np.sin(2 * np.pi * 1000 * t)).astype(np.int16)

with wave.open(TONE_WAV, "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(playback_rate)
    wf.writeframes(tone.tobytes())

print(f"\n🔊 [1/2] Playing 1kHz test tone for 4 seconds at {playback_rate} Hz...")
print("   >> Check phone sound meter app at 1.0m to verify >= 75 dBA SPL!")

try:
    sd.play(tone, samplerate=playback_rate, device=target_device)
    sd.wait()
    print("✅ Tone played successfully through USB Speaker!")
except Exception as e:
    print(f"sounddevice error: {e}. Falling back to ALSA plughw:")
    subprocess.run(f"aplay -D plughw:3,0 -q {TONE_WAV}", shell=True)

# 3. Speech Playback (if available from Test 1)
if os.path.exists(SPEECH_WAV):
    print("\n🔊 [2/2] Playing back your 1-meter speech recording...")
    try:
        with wave.open(SPEECH_WAV, "rb") as wf:
            sr = wf.getframerate()
            frames = wf.readframes(wf.getnframes())
            data = np.frombuffer(frames, dtype=np.int16)
        sd.play(data, samplerate=sr, device=target_device)
        sd.wait()
        print("✅ Speech playback completed.")
    except Exception as e:
        subprocess.run(f"aplay -D plughw:3,0 -q {SPEECH_WAV}", shell=True)
else:
    print("\nℹ️ [2/2] Skipped speech playback (no speech file recorded in Test 1).")

---
## Test 3: Power, Voltage Stability & CPU Stress Test (`vcgencmd`)
Tests power supply stability and thermal throttling:
- Reads `vcgencmd get_throttled` for past and present under-voltage flags ($< 4.63\text{V}$).
- Measures core voltage, PMIC 5V rail (Raspberry Pi 5), and SoC temperature.
- Executes a 5-second **100% CPU multi-core stress test** to ensure the power supply does not dip under full load.

In [ ]:
import multiprocessing

THROTTLED_FLAGS = {
    0: "Under-voltage currently detected (< 4.63V)",
    1: "ARM frequency currently capped",
    2: "Currently throttled",
    3: "Soft temperature limit currently active",
    16: "Under-voltage has occurred since last boot",
    17: "ARM frequency capping has occurred",
    18: "Throttling has occurred since last boot",
    19: "Soft temperature limit has occurred",
}

def read_power_metrics():
    throttled_raw = run_cmd("vcgencmd get_throttled")
    volts_core = run_cmd("vcgencmd measure_volts core")
    temp = run_cmd("vcgencmd measure_temp")
    pmic_adc = run_cmd("vcgencmd pmic_read_adc")
    ext5v = None
    if "EXT5V_V" in pmic_adc:
        for line in pmic_adc.splitlines():
            if "EXT5V_V" in line:
                ext5v = line.strip()
                break
    return throttled_raw, volts_core, temp, ext5v

def decode_throttled(throttled_str):
    if "=" not in throttled_str:
        return -1, ["vcgencmd command unavailable (ensure running directly on Raspberry Pi)"]
    val = int(throttled_str.split("=")[1], 16)
    issues = [desc for bit, desc in THROTTLED_FLAGS.items() if (val & (1 << bit))]
    return val, issues

print("=== 1. Baseline Power & Thermal Metrics (Idle) ===")
t_idle, v_idle, temp_idle, ext5v_idle = read_power_metrics()
print(f"SoC Temperature   : {temp_idle}")
print(f"Core Voltage      : {v_idle}")
if ext5v_idle:
    print(f"PMIC 5V Rail      : {ext5v_idle}")
print(f"Throttled Status  : {t_idle}")

print("\n=== 2. Running 5-Second Multi-Core CPU Stress Test ===")
def cpu_burn(duration):
    end = time.time() + duration
    x = 0.0001
    while time.time() < end:
        x = math.sqrt(x * x + 1.2345)

procs = [multiprocessing.Process(target=cpu_burn, args=(5,)) for _ in range(multiprocessing.cpu_count())]
for p in procs:
    p.start()

time.sleep(3.5)  # Sample in peak current draw
t_load, v_load, temp_load, ext5v_load = read_power_metrics()

for p in procs:
    p.join()

print(f"SoC Temp under Load    : {temp_load}")
print(f"Core Voltage under Load: {v_load}")
if ext5v_load:
    print(f"PMIC 5V under Load     : {ext5v_load}")
print(f"Throttled Status (Load): {t_load}")

code_val, issues = decode_throttled(t_load)
print("\n=== Power Stability Verdict ===")
if code_val == 0:
    print("✅ PASS (0x0): Power supply is completely stable. Zero under-voltage detected even under 100% CPU load!")
elif code_val > 0:
    print("⚠️ WARNING: Throttling / Power events detected:")
    for issue in issues:
        print(f"   - {issue}")
else:
    print("❌ Could not query vcgencmd.")